# Evaluate the model - Matterport

Use the JuniperMapper fieldwork data to validate the trained model. To evaluate the overall model performance, three metrics were used:

1. Precision metric: assess how precise the model is in delineating the shrubs
2. Recall: evaluate how accurate the model is in recalling the shrubs
3. F1-score: examine the ability of the model to maintain the trade-off precision–recall.

$$
Precision = \frac{TP}{TP+FP}
$$

$$
Recall = \frac{TP}{TP+FN}
$$

$$
F1 = \frac{2PR}{P+R}
$$

For Mask R-CNN, there are two useful interpretations of Precision, Recall, and F1:

1. Pixel-level segmentation metrics: measure how precisely the predicted masks delineate shrub pixels.
2. Instance-level detection metrics: measure whether individual shrubs are found, using a mask-IoU threshold.

This code reporting both, although the reference paper ([Khaldi et al. 2024](https://doi.org/10.1016/j.jag.2024.104191)) report only instance-level detection.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'bin'))
import mrcnn.model as modellib
from shrubs_matterport import ShrubDataset
from shrubs_matterport import BestModelShrubInferenceConfig
from shrubs_matterport import load_ground_truths
from shrubs_matterport import predict_dataset
from shrubs_common import evaluate_pixel_metrics
from shrubs_common import evaluate_instance_iou
from shrubs_common import evaluate_instance_siou
from shrubs_common import confidence_thresholds
from shrubs_common import write_sweep
from shrubs_common.plotting import plot_performance

## Configurations

In [ ]:
%matplotlib inline 

ROOT = os.path.join(os.path.dirname(os.getcwd()))

# Directory to save logs and trained model
MODEL_DIR = os.path.join(ROOT, "data/logs/matterport")

# Directory with test data
VAL_DIR_PI = os.path.join(
    ROOT,
    "data/JuniperMapper", "Photo_Interpretation_Data/Test"
)

VAL_DIR_FW = os.path.join(
    ROOT,
    "data/JuniperMapper", "Field_Work_Data/External_Val_Data"
)

# Directory to save evaluation stats
STATS_DIR = os.path.join(ROOT, "results/evaluation/model_matterport")

In [ ]:
# Validation datasets
dataset_val_pi = ShrubDataset()
dataset_val_pi.load_dataset(
    os.path.join(VAL_DIR_PI, "Images"),
    os.path.join(VAL_DIR_PI, "masks_points.json")
)
dataset_val_pi.prepare()

dataset_val_fw = ShrubDataset()
dataset_val_fw.load_dataset(
    os.path.join(VAL_DIR_FW, "Images"),
    os.path.join(VAL_DIR_FW, "masks_points.json")
)
dataset_val_fw.prepare()

In [ ]:
# Get the path to best model configuration
best_meta_path = os.path.join(MODEL_DIR, "best_model_configuration.json")

config = BestModelShrubInferenceConfig(best_meta_path)

# Create model in testing mode
model = modellib.MaskRCNN(
    mode="inference",
    config=config,
    model_dir=MODEL_DIR
)

In [ ]:
print("Loading weights from ", config.best_model_path)
model.load_weights(config.best_model_path, by_name=True)

## Inference

Choose the dataset to evaluate, then run inference once. Every metric below (pixel, IoU and S-IoU) is computed with the shared `shrubs_common` code, the same used for Detectron2 and SAM 3, from this single prediction cache.

Run the following chunk again whenever `dataset_obj` is changed.

In [ ]:
dataset_name = "shrubs_test_pi"
dataset_obj = dataset_val_pi
out_dir = Path(STATS_DIR) / dataset_name

ground_truths = load_ground_truths(dataset_obj)
predictions = predict_dataset(model, dataset_obj, verbose=False)

## Pixel-level metrics

Combine all shrub-instance masks in an image into one binary shrub mask. Then count the three metrics to assess overall model performance. `TP`, `FP`, and `FN` are numbers of pixels.

In [ ]:
pixel_results = evaluate_pixel_metrics(
    ground_truths,
    predictions,
    class_id=1,
    confidence_threshold=0.5,
)

print(f"Pixel-level evaluation for {dataset_name}")
print(f"Precision: {pixel_results['precision']:.4f}")
print(f"Recall:    {pixel_results['recall']:.4f}")
print(f"F1-score:  {pixel_results['f1']:.4f}")
print(f"TP pixels: {pixel_results['tp_pixels']}")
print(f"FP pixels: {pixel_results['fp_pixels']}")
print(f"FN pixels: {pixel_results['fn_pixels']}")

These metrics answer:

* **Pixel Precision:** Of all pixels labelled as shrubs, how many really belong to shrubs?
* **Pixel Recall:** Of all annotated shrub pixels, how many were recovered?
* **Pixel F1:** How well does the model balance over-segmentation and under-segmentation?

## Instance-level metrics

The IoU metric supports two definitions, selected with `iou_metric_definition`:

* **`"standard"`** (COCO-style): predictions are processed by descending confidence and each is matched to the unmatched ground-truth shrub with the highest mask IoU, provided that IoU reaches the threshold (`shrubs_common.one_to_one_matches()`). Each label can be matched at most once, so TP + FN always equals the number of labels, and Recall = matched labels / all labels.
* **`"paper"`** ([Khaldi et al. 2024](https://doi.org/10.1016/j.jag.2024.104191)): predictions and labels are evaluated independently against their best-matching counterpart (`shrubs_common.best_iou_matches()`). A prediction is a TP when its best IoU with any label reaches the threshold, and a label is an FN when its best IoU with any prediction does not. Labels are not consumed, so duplicated predictions of one shrub are all TPs, and Recall = TP / (TP + FN).

Both use mask IoU rather than bounding-box IoU, so the quality of the predicted segmentation decides whether a shrub was detected correctly.

Note: `predict_dataset` converts Mask R-CNN outputs to the shared format: instance masks in `masks` (HxWxN), class IDs in `class_ids`, confidence scores in `scores`, and XYXY boxes in `boxes` (Mask R-CNN's `rois` are YXYX).

### Iterate between several confident thresholds

Use a declared pair of thresholds.

* The **confidence threshold** controls which predicted instances are included.
* The **mask-IoU threshold** controls how closely a predicted mask must overlap an annotation to count as a correct shrub instance. Higher IoU thresholds impose stricter delineation requirements.

The sweep is evaluated at every IoU threshold in `ev_metric_thresholds` and each result is saved to a csv file.

In [ ]:
ev_metric_thresholds = [0.50, 0.75]
thresholds = confidence_thresholds(None)

# "standard" (COCO-style one-to-one) or "paper" (Khaldi et al. 2024).
# Every file is saved flat inside `out_dir`; the definition is encoded in the
# IoU csv and figure names, so both definitions can share one folder.
iou_metric_definition = "paper"
iou_prefix = "iou_paper" if iou_metric_definition == "paper" else "iou"
iou_label = "IoU (paper)" if iou_metric_definition == "paper" else "IoU"
plot_prefix = "evaluation_paper" if iou_metric_definition == "paper" else "evaluation"

In [ ]:
for ev_metric_threshold in ev_metric_thresholds:
    write_sweep(
        Path(out_dir, f"{iou_prefix}_threshold_{round(ev_metric_threshold * 100)}.csv"),
        metric_name="IoU",
        evaluation_threshold=ev_metric_threshold,
        thresholds=thresholds,
        evaluator=lambda confidence, t=ev_metric_threshold: evaluate_instance_iou(
            ground_truths, predictions, class_id=1,
            confidence_threshold=confidence, iou_threshold=t,
            metric_definition=iou_metric_definition,
        ),
    )

The paper where JuniperMapper dataset is evaluated ([Khaldi et al. 2024](https://doi.org/10.1016/j.jag.2024.104191)) used also a S-IoU metric to assess the instance-level metrics. The S-IoU:

> evaluates the proportion of shrubs' areas being detected by the model since they divide the intersection by the ground truth area and they use all matching ground truth shrubs instead of the best matching shrub. This metric evaluates the predictions and labels differently. The first equation was used to identify the number of TPs and FPs, while the second was used to identify the number of FNs.

$$
\mathrm{S\text{-}IoU}(p, S_{l_{\mathrm{match}}})
=
\frac{
\operatorname{Area}\left(
p \cap \bigcup_{k=1}^{\left|S_{l_{\mathrm{match}}}\right|} l_k
\right)
}{
\operatorname{Area}\left(
\bigcup_{k=1}^{\left|S_{l_{\mathrm{match}}}\right|} l_k
\right)
}
$$

$$
\mathrm{S\text{-}IoU}(l, S_{p_{\mathrm{match}}})
=
\frac{
\operatorname{Area}\left(
l \cap \bigcup_{k=1}^{\left|S_{p_{\mathrm{match}}}\right|} p_k
\right)
}{
\operatorname{Area}(l)
}
$$

where: $S_{p_{\mathrm{match}}}$ the set of all matching predictions. $S_{l_{\mathrm{match}}}$ is the set of all matching labels. $l_k$ and $p_k$ are the label and prediction of id $k$, respectively.

The shared metric computes the two independent calculations required by S-IoU metric:

- $\mathrm{S\text{-}IoU}(p, S_{l_{\mathrm{match}}})$: determines whether each prediction is TP or FP.
- $\mathrm{S\text{-}IoU}(l, S_{p_{\mathrm{match}}})$: determines whether each ground-truth instance is an FN.

The function `shrubs_common.compute_s_iou_matches` is able to match one label to several predictions and several labels to one prediction. It defines matching instances as same-class masks with positive overlap.

> Note: As in the paper, recall uses the prediction-directed TP count together with the label-directed FN count. TP counts predictions and FN counts labels, so TP + FN need not equal the number of labels.

Implemented interpretation (`shrubs_common.evaluate_instance_siou`, following the paper):

| | |
|------------|----------------|
|Matching set | All same-class masks with at least `minimum_intersection_pixels` (1 in this notebook) intersecting pixels.|
|S-IoU threshold|Applied after unioning matching masks.|
|Prediction evaluation|S-IoU(p, matching labels) >= threshold -> TP; Otherwise -> FP |
|Label evaluation|S-IoU(l, matching predictions) < threshold -> FN|
|Precision | TP / (TP + FP)|
|Recall | TP / (TP + FN) |
|F1 | Harmonic mean of the precision and recall above |

In [ ]:
for ev_metric_threshold in ev_metric_thresholds:
    write_sweep(
        Path(out_dir, f"siou_threshold_{round(ev_metric_threshold * 100)}.csv"),
        metric_name="S-IoU",
        evaluation_threshold=ev_metric_threshold,
        thresholds=thresholds,
        evaluator=lambda confidence, t=ev_metric_threshold: evaluate_instance_siou(
            ground_truths, predictions, class_id=1,
            confidence_threshold=confidence, siou_threshold=t,
            minimum_intersection_pixels=1,
        ),
    )

### Plot the results

Note: Because the model performs binary classification between background and shrub, a region is retained as a shrub only when its shrub probability exceeds its background probability, which implicitly imposes a minimum confidence of approximately 0.5. Consequently, setting the explicit confidence threshold between 0 and 0.5 does not change the retained predictions or the resulting precision, recall, and F1-score. Variations in these metrics are therefore observed only at thresholds above 0.5.

In [ ]:
csv_files = {
    ev_metric_threshold: {
        iou_label: Path(out_dir, f"{iou_prefix}_threshold_{round(ev_metric_threshold * 100)}.csv"),
        "S-IoU": Path(out_dir, f"siou_threshold_{round(ev_metric_threshold * 100)}.csv"),
    }
    for ev_metric_threshold in ev_metric_thresholds
}

# Store the graphics next to the csv files
plot_performance(csv_files, out_dir, show=True, file_prefix=plot_prefix)